In [1]:
%%pyspark project.spark.compatibility
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType, MapType
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
import boto3

spark = (
    SparkSession.builder
    .appName("rule_validation_fraud")
    .config("spark.executor.memory", "18g")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "300")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

Executing for connection type: SPARK_GLUE, connection name: project.spark.compatibility
Creating Glue session...
Create session for connection: project.spark.compatibility


'Session ae2xwmv6edkt56-1c5310cf-4402-41b4-920c-75d822e6fc30 has been created.'

<sagemaker_studio_dataengineering_sessions.sagemaker_base_session_manager.common.debugging_utils.SessionInfoTableDisplay object>

Session created for connection: project.spark.compatibility.

Connection: project.spark.compatibility | Run start time: 2026-06-08 16:17:33.394080 | Run duration : 0:02:26.913596s.


In [2]:
%%pyspark project.spark.compatibility
# fraud_cibc_rogers = spark.read.parquet('s3://enstream-lake-gold-dev/poc/analysis/imf_real_data_validation/run_id=imf_real_data_validation_bugfixed_20260514_193428/extended_k_analysis/run_id=imf_extended_k_20260515_142025/scored/phase1_scored.parquet/')
# fraud_nbc = spark.read.parquet('s3://enstream-lake-gold-dev/poc/analysis/imf_real_data_validation/run_id=imf_real_data_validation_bugfixed_20260514_193428/extended_k_analysis/run_id=imf_extended_k_20260515_142025/scored/nbc_scored.parquet/')
# frauds = fraud_cibc_rogers.unionByName(fraud_nbc)

Connection: project.spark.compatibility | Run start time: 2026-06-08 16:20:00.312625 | Run duration : 0:00:00.546619s.


In [3]:
%%pyspark project.spark.compatibility
# FRAUDS_OUT = "s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2/frauds_raw"
# frauds.write.mode("overwrite").parquet(FRAUDS_OUT)
# print(f"saved frauds → {FRAUDS_OUT}")
# print(f"rows: {frauds.count():,}, cols: {len(frauds.columns)}")


Connection: project.spark.compatibility | Run start time: 2026-06-08 16:20:00.863067 | Run duration : 0:00:00.034462s.


In [4]:
%%pyspark project.spark.compatibility
# frauds = spark.read.parquet("s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2/frauds_raw/")

Connection: project.spark.compatibility | Run start time: 2026-06-08 16:20:00.900905 | Run duration : 0:00:00.034315s.


In [5]:
%%pyspark project.spark.compatibility
FAM = "s3://enstream-lake-gold-dev/poc/features"

# FAMILIES = [
#     "account_suspension", "device_change", "enstream_api_call_generic",
#     "enstream_partner", "mno_port", "phone_number_change",
#     "sim_change", "stability", "state_transitions",
# ]
# STATE_FAM = "state_transitions"

# def _load_one(name, suffix, version, months_list):
#     base = f"{FAM}/feature_set=time_aware_event_based_behavioural_customer_{name}{suffix}/version={version}/"
#     paths = [f"{base}dt={m}/" for m in months_list]          # <- back to directory paths (spacing irrelevant)
#     return (spark.read
#             .option("header", "true")
#             .option("inferSchema", "true")
#             # .option("pathGlobFilter", "part-*")              # <- filename filter, not part of the path URI
#             .csv(paths))

# def load_all_families(mode, months_list, months_state_list):
#     """mode='eval'(fraud) or 'normal'. Returns dict: family -> df (already deduped)"""
#     if mode == "eval":
#         suffix, versions = "_eval", ["phase1-v1.0", "NBC-v1.0"]
#     elif mode == "normal":
#         suffix, versions = "", ["1"]
#     else:
#         raise ValueError(f"unknown mode: {mode}")

#     out = {}
#     for fam in FAMILIES:
#         m = months_state_list if fam == STATE_FAM else months_list
#         df = None
#         for v in versions:
#             part = _load_one(fam, suffix, v, m)
#             df = part if df is None else df.unionByName(part)
#         # <- dedup each family first (by unique observation key)
#         dedup_keys = [k for k in ["customer_id", "reference_id", "reference_ts"] if k in df.columns]
#         df = df.dropDuplicates(dedup_keys)
#         out[fam] = df
#     return out


Connection: project.spark.compatibility | Run start time: 2026-06-08 16:20:00.939283 | Run duration : 0:00:01.275060s.


In [6]:
%%pyspark project.spark.compatibility
# months_fraud = [
#     "2024-10-01 00:00:00","2024-11-01 00:00:00","2024-12-01 00:00:00",
#     "2025-01-01 00:00:00","2025-02-01 00:00:00","2025-03-01 00:00:00",
#     "2025-04-01 00:00:00","2025-05-01 00:00:00","2025-06-01 00:00:00",
#     "2025-07-01 00:00:00","2025-08-01 00:00:00","2025-09-01 00:00:00",
#     "2025-10-01 00:00:00",
# ]
# months_state_fraud = [
#     "2024-10-01 00:00:00","2024-11-01 00:00:00","2024-12-01 00:00:00",
#     "2025-01-01 00:00:00","2025-02-01 00:00:00","2025-03-01 00:00:00",
#     "2025-04-01 00:00:00","2025-05-01 00:00:00",
# ]
# fraud_feats = load_all_families("eval", months_fraud, months_state_fraud)


Connection: project.spark.compatibility | Run start time: 2026-06-08 16:20:02.218028 | Run duration : 0:00:00.569417s.


In [7]:
%%pyspark project.spark.compatibility
# ID_COLS = {"customer_id", "reference_id", "reference_ts", "from_ts", "to_ts", "dt"}

# def feat_add_dt(feat_df):
#     return feat_df.withColumn('dt', F.to_date(F.to_timestamp('reference_ts')).cast("timestamp"))

# def join_full_feat(base_df, feat_df):
#     feat_df = feat_add_dt(feat_df)
#     feat_df = feat_df.dropDuplicates(["customer_id", "dt"])        # prevent row fan-out
#     shared_cols = list(set(base_df.columns) & set(feat_df.columns))   # 41 overlapping cols also go into key -> merge
#     return base_df.join(feat_df, on=shared_cols, how="left")

# # Note dt type: frauds.dt is string, cast to timestamp to align with feature
# base_df = (frauds
#            .withColumn("dt", F.to_timestamp(F.col("dt")))         # string -> timestamp (consistent with feature)
#            .filter(F.col("dt") >= F.lit("2024-10-01")))

# for feat_df in fraud_feats.values():
#     base_df = join_full_feat(base_df, feat_df)

# final_frauds = base_df.dropDuplicates(["customer_id", "dt", "fraud_type", "partner"])
# print(f"final_frauds: {final_frauds.count():,} rows, {len(final_frauds.columns)} cols")

# print(f"frauds raw:           {frauds.count():,}")
# print(f"frauds (from 2024-10): {base_df.count():,}")

# for feat_df in fraud_feats.values():
#     base_df = join_full_feat(base_df, feat_df)

# # dedup by the true fraud grain (reference_ts is gone, use dt instead)
# final_frauds = base_df.dropDuplicates(["customer_id", "dt", "fraud_type", "partner"])
# print(f"final_frauds (join+dedup): {final_frauds.count():,}, cols: {len(final_frauds.columns)}")

Connection: project.spark.compatibility | Run start time: 2026-06-08 16:20:02.791372 | Run duration : 0:00:00.039784s.


In [8]:
%%pyspark project.spark.compatibility
# output_path = "s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2_fraud"
# final_frauds.write \
#     .mode("overwrite") \
#     .parquet(output_path)

Connection: project.spark.compatibility | Run start time: 2026-06-08 16:20:02.834584 | Run duration : 0:00:00.042662s.


In [9]:
%%pyspark project.spark.compatibility
final_frauds = spark.read.parquet("s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2_fraud")


Connection: project.spark.compatibility | Run start time: 2026-06-08 16:20:02.880507 | Run duration : 0:00:31.036779s.


In [10]:
%%pyspark project.spark.compatibility
BASE = ("s3://enstream-lake-gold-dev/poc/analysis/imf_real_data_validation/"
        "run_id=imf_real_data_validation_bugfixed_default_available_feature_requests_20260603_172459/"
        "full_scored_per_k_export/run_id=full_scored_per_k_export_20260605_160344/scored_full")

# nonfraud: 10 batch CSVs in the folder, reading the folder = union
nonfraud_oct = (spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("pathGlobFilter", "*.csv")        # read csv only, skip the manifest
            .csv(f"{BASE}/nonfraud_batches/"))
print("nonfraud:", nonfraud_oct.count(), "rows,", len(nonfraud_oct.columns), "cols")


nonfraud: 1000000 rows, 124 cols

Connection: project.spark.compatibility | Run start time: 2026-06-08 16:20:33.922511 | Run duration : 0:00:27.935330s.


In [42]:
%%pyspark project.spark.compatibility
from pyspark.sql import functions as F

# ---- 1) check option5 directly on the just-loaded nonfraud_oct (column name uses double underscore __ here) ----
opt5 = [c for c in nonfraud_oct.columns
        if "option5_reservoir_sampled_imf" in c and c.endswith(("raw_score", "anomaly_score"))]
print("option5 score columns in nonfraud_oct:", opt5)

for col in opt5:
    nd = nonfraud_oct.select(col).distinct().count()
    print(f"\n===== {col} | distinct = {nd} =====")
    (nonfraud_oct.groupBy(F.round(F.col(col), 4).alias("value"))
        .count().orderBy(F.desc("count")).show(10, truncate=False))

# ---- 2) while we are at it, scan all score columns to see if any in source is non-constant ----
score_like = [c for c in nonfraud_oct.columns
              if any(k in c.lower() for k in ["score", "imf", "anomaly"]) and "label" not in c.lower()]
print("\n===== approx distinct of all score columns in nonfraud_oct (>1 means a real distribution) =====")
nonfraud_oct.agg(*[F.approx_count_distinct(c).alias(c) for c in score_like]).show(truncate=False, vertical=True)


nonfraud_oct 里的 option5 分数列: ['option5_reservoir_sampled_imf__raw_score', 'option5_reservoir_sampled_imf__anomaly_score']

===== option5_reservoir_sampled_imf__raw_score | distinct = 1 =====
+------+-------+
|value |count  |
+------+-------+
|0.1642|1000000|
+------+-------+


===== option5_reservoir_sampled_imf__anomaly_score | distinct = 1 =====
+-----+-------+
|value|count  |
+-----+-------+
|0.5  |1000000|
+-----+-------+


===== 所有 score 类列在 nonfraud_oct 的 approx distinct（>1 才有真分布）=====
-RECORD 0---------------------------------------------
 option3_additive_monthly_imf__raw_score        | 1   
 option3_additive_monthly_imf__anomaly_score    | 1   
 option3b_online_leaf_update_imf__raw_score     | 1   
 option3b_online_leaf_update_imf__anomaly_score | 1   
 option4_rolling_window_imf__raw_score          | 1   
 option4_rolling_window_imf__anomaly_score      | 1   
 option5_reservoir_sampled_imf__raw_score       | 1   
 option5_reservoir_sampled_imf__anomaly_score   | 1

Connection

In [41]:
%%pyspark project.spark.compatibility
from pyspark.sql import functions as F

RAW  = "option5_reservoir_sampled_imf__raw_score"
ANOM = "option5_reservoir_sampled_imf__anomaly_score"

score_cols = [RAW, ANOM]

summary_exprs = []
for c in score_cols:
    summary_exprs += [
        F.count(c).alias(f"{c}_count"),
        F.min(c).alias(f"{c}_min"),
        F.max(c).alias(f"{c}_max"),
        F.avg(c).alias(f"{c}_avg"),
        F.stddev(c).alias(f"{c}_std"),
    ]

nonfraud_oct.agg(*summary_exprs).show(truncate=False)

+----------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------------+------------------------------------------------+------------------------------------------------+------------------------------------------------+------------------------------------------------+
|option5_reservoir_sampled_imf__raw_score_count|option5_reservoir_sampled_imf__raw_score_min|option5_reservoir_sampled_imf__raw_score_max|option5_reservoir_sampled_imf__raw_score_avg|option5_reservoir_sampled_imf__raw_score_std|option5_reservoir_sampled_imf__anomaly_score_count|option5_reservoir_sampled_imf__anomaly_score_min|option5_reservoir_sampled_imf__anomaly_score_max|option5_reservoir_sampled_imf__anomaly_score_avg|option5_reservoir_sampled_imf__anomaly_score_std|
+----------------------------------------------+

In [ ]:
%%pyspark project.spark.compatibility
+----------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------------+------------------------------------------------+------------------------------------------------+------------------------------------------------+------------------------------------------------+
|option5_reservoir_sampled_imf__raw_score_count|option5_reservoir_sampled_imf__raw_score_min|option5_reservoir_sampled_imf__raw_score_max|option5_reservoir_sampled_imf__raw_score_avg|option5_reservoir_sampled_imf__raw_score_std|option5_reservoir_sampled_imf__anomaly_score_count|option5_reservoir_sampled_imf__anomaly_score_min|option5_reservoir_sampled_imf__anomaly_score_max|option5_reservoir_sampled_imf__anomaly_score_avg|option5_reservoir_sampled_imf__anomaly_score_std|
+----------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------------+------------------------------------------------+------------------------------------------------+------------------------------------------------+------------------------------------------------+
|1000000                                       |0.16421514749526978                         |0.16421514749526978                         |0.16421514749526978                         |0.0                                         |1000000                                           |0.5                                             |0.5                                             |0.5                                             |0.0                                             |
+----------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------+--------------------------------------------------+------------------------------------------------+------------------------------------------------+------------------------------------------------+------------------------------------------------+


In [11]:
%%pyspark project.spark.compatibility
nonfraud_oct.printSchema()


root
 |-- customer_id: string (nullable = true)
 |-- dt: date (nullable = true)
 |-- record_source: string (nullable = true)
 |-- actual_label: integer (nullable = true)
 |-- option3_additive_monthly_imf__raw_score: double (nullable = true)
 |-- option3_additive_monthly_imf__anomaly_score: double (nullable = true)
 |-- option3b_online_leaf_update_imf__raw_score: double (nullable = true)
 |-- option3b_online_leaf_update_imf__anomaly_score: double (nullable = true)
 |-- option4_rolling_window_imf__raw_score: double (nullable = true)
 |-- option4_rolling_window_imf__anomaly_score: double (nullable = true)
 |-- option5_reservoir_sampled_imf__raw_score: double (nullable = true)
 |-- option5_reservoir_sampled_imf__anomaly_score: double (nullable = true)
 |-- final_label__option3_additive_monthly_imf__top_fixed_50: integer (nullable = true)
 |-- final_label__option3_additive_monthly_imf__top_fixed_100: integer (nullable = true)
 |-- final_label__option3_additive_monthly_imf__top_pct_0p01: int

In [12]:
%%pyspark project.spark.compatibility
months = ["2024-10-01"]
months_state = months
FEAT_SUFFIX  = ""           
FEAT_VERSION = "2" 

def normal_load_family(name, months_list=months):
    base = f"{FAM}/feature_set=time_aware_event_based_behavioural_customer_{name}{FEAT_SUFFIX}/version={FEAT_VERSION}/"
    paths = [f"{base}dt={m}/" for m in months_list]
    return spark.read.option("header", "true").option("inferSchema", "true").csv(paths)
    
oct_df_account_suspension  = normal_load_family("account_suspension")
oct_df_device_change       = normal_load_family("device_change")
oct_df_api_call            = normal_load_family("enstream_api_call_generic")
oct_df_partner             = normal_load_family("enstream_partner")
oct_df_mno_port            = normal_load_family("mno_port")
oct_df_phone_number_change = normal_load_family("phone_number_change")
oct_df_sim_change          = normal_load_family("sim_change")
oct_df_stability           = normal_load_family("stability")
oct_df_state_transitions   = normal_load_family("state_transitions", months_list=months_state)


Connection: project.spark.compatibility | Run start time: 2026-06-08 16:21:04.614497 | Run duration : 0:02:00.814119s.


In [13]:
%%pyspark project.spark.compatibility
nonfraud_oct.count()



1000000
Connection: project.spark.compatibility | Run start time: 2026-06-08 16:23:05.810908 | Run duration : 0:00:04.783561s.


In [14]:
%%pyspark project.spark.compatibility
def convert_str_ts_to_dt_to_ts(feat_df):
    feat_df = feat_df.withColumn('dt', F.to_date(F.to_timestamp('reference_ts')).cast("timestamp"))
    return feat_df

def join_full_feat(base_df, feat_df):
    feat_df = convert_str_ts_to_dt_to_ts(feat_df)
    base_cols = list(base_df.columns)
    feat_cols = list(feat_df.columns)
    shared_cols = list(set(base_cols) & set(feat_cols))#[feat_col for feat_col in base_cols for feat_col in feat_cols]
    extended_df = base_df.join(feat_df, on = shared_cols, how = 'left')
    return extended_df
        
feat_dfs = [oct_df_account_suspension, oct_df_device_change, oct_df_api_call,
            oct_df_partner, oct_df_mno_port, oct_df_phone_number_change,
            oct_df_sim_change, oct_df_stability, oct_df_state_transitions]

base_df = nonfraud_oct
for feat_df in feat_dfs:
    base_df = join_full_feat(base_df, feat_df)


Connection: project.spark.compatibility | Run start time: 2026-06-08 16:23:10.598657 | Run duration : 0:00:03.832552s.


In [15]:
%%pyspark project.spark.compatibility
base_df.count()


1000000
Connection: project.spark.compatibility | Run start time: 2026-06-08 16:23:14.435648 | Run duration : 0:01:23.525473s.


In [16]:
%%pyspark project.spark.compatibility
final_nonfraud_oct = base_df.dropDuplicates(['customer_id', 'reference_ts'])
final_nonfraud_oct.count()



1000000
Connection: project.spark.compatibility | Run start time: 2026-06-08 16:24:37.965267 | Run duration : 0:00:09.990710s.


In [17]:
%%pyspark project.spark.compatibility
final_nonfraud_oct = final_nonfraud_oct.withColumn('fraud_type', F.lit('non_fraud'))


Connection: project.spark.compatibility | Run start time: 2026-06-08 16:24:47.960095 | Run duration : 0:00:01.822154s.


In [18]:
%%pyspark project.spark.compatibility
row_count = final_nonfraud_oct.count()
col_count = len(final_nonfraud_oct.columns)
print(f"Rows: {row_count}")
print(f"Columns: {col_count}")

Rows: 1000000
Columns: 461

Connection: project.spark.compatibility | Run start time: 2026-06-08 16:24:49.787057 | Run duration : 0:00:08.411753s.


In [19]:
%%pyspark project.spark.compatibility
output_path = "s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2_oct_nonfraud"
final_nonfraud_oct.write \
    .mode("overwrite") \
    .parquet(output_path)




Connection: project.spark.compatibility | Run start time: 2026-06-08 16:24:58.203475 | Run duration : 0:02:56.579985s.


In [20]:
%%pyspark project.spark.compatibility
final_nonfraud_oct = spark.read.parquet('s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2_oct_nonfraud')



Connection: project.spark.compatibility | Run start time: 2026-06-08 16:27:54.787570 | Run duration : 0:00:02.386922s.


In [21]:
%%pyspark project.spark.compatibility
frauds_val = final_frauds.filter(F.to_date(F.col("dt")) == F.lit("2024-10-01"))
print(f"frauds_val (2024-10): {frauds_val.count():,}")

# Fix: normal's IMF score column uses double __, fraud uses single _; the name mismatch is why they got filled NULL.
# before union, rename normal's columns to single _ to align with fraud -> merge into one column with values on both sides
for old, new in {
    "option5_reservoir_sampled_imf__anomaly_score": "option5_reservoir_sampled_imf_anomaly_score",
    "option5_reservoir_sampled_imf__raw_score":     "option5_reservoir_sampled_imf_raw_score",
}.items():
    if old in final_nonfraud_oct.columns:
        final_nonfraud_oct = final_nonfraud_oct.withColumnRenamed(old, new)

all_validation = frauds_val.unionByName(final_nonfraud_oct, allowMissingColumns=True)
print(f"all_validation: {all_validation.count():,}")

frauds_val (2024-10): 1,588
all_validation: 1,001,588

Connection: project.spark.compatibility | Run start time: 2026-06-08 16:27:57.178536 | Run duration : 0:00:05.633995s.


In [22]:
%%pyspark project.spark.compatibility
output_path = "s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2_validation"
all_validation.write \
    .mode("overwrite") \
    .parquet(output_path)




Connection: project.spark.compatibility | Run start time: 2026-06-08 16:28:02.816412 | Run duration : 0:00:22.967803s.


In [23]:
%%pyspark project.spark.compatibility
# ─── RULES ─────────────────────────────────────────────────────────
# format: feature → {fraud: (op, val), normal: (op, val)}
# uncertain = NOT fraud AND NOT normal AND value IS NOT NULL

RULES = {
    "cust_account_suspension_days_for_last_5_events": {"fraud": (">=", 184), "normal": ("<", 158.5)},
    "cust_account_suspension_k_event_velocity_5": {"fraud": ("<=", 0.0272), "normal": (">", 0.0323)},
    "cust_mno_port_cumulative_event_slope": {"fraud": ("<=", 0.0036), "normal": (">", 0.0102)},
    "cust_mno_port_days_for_last_3_events": {"fraud": ("<=", 6), "normal": (">", 53)},
    "cust_mno_port_interval_cv": {"fraud": (">=", 1.7321), "normal": ("<", 1.4142)},
    "cust_mno_port_interval_max": {"fraud": ("<=", 4), "normal": (">", 29)},
    "cust_mno_port_interval_mean": {"fraud": ("<=", 4), "normal": (">", 23)},
    "cust_mno_port_interval_median": {"fraud": ("<=", 3), "normal": (">", 80)},
    "cust_mno_port_interval_min": {"fraud": ("<=", 3), "normal": (">", 80)},
    "cust_mno_port_interval_stddev": {"fraud": ("<=", 17), "normal": (">", 46.7654)},
    "cust_mno_port_inverse_mean_interarrival_velocity": {"fraud": ("<=", 0.0049), "normal": (">", 0.0113)},
    "cust_mno_port_k_event_velocity_3": {"fraud": (">=", 0.5), "normal": ("<", 0.037)},
    "cust_mno_port_monthly_cnt_std": {"fraud": ("<=", 0), "normal": (">", 0.1)},
    "cust_phone_number_change_cumulative_event_slope": {"fraud": (">=", 0.5), "normal": ("<", 0.3333)},
    "cust_phone_number_change_days_for_last_3_events": {"fraud": ("<=", 1), "normal": (">", 2)},
    "cust_phone_number_change_interval_cv": {"fraud": (">=", 1.4142), "normal": ("<", 1.1705)},
    "cust_phone_number_change_interval_max": {"fraud": ("<=", 0), "normal": (">", 3)},
    "cust_phone_number_change_interval_mean": {"fraud": ("<=", 0), "normal": (">", 3)},
    "cust_phone_number_change_interval_median": {"fraud": ("<=", 0), "normal": (">", 3)},
    "cust_phone_number_change_interval_min": {"fraud": ("<=", 0), "normal": (">", 3)},
    "cust_phone_number_change_interval_stddev": {"fraud": ("<=", 0), "normal": (">", 1.4142)},
    "cust_phone_number_change_inverse_mean_interarrival_velocity": {"fraud": (">=", 1), "normal": ("<", 0.5)},
    "cust_phone_number_change_k_event_velocity_3": {"fraud": (">=", 1.5), "normal": ("<", 0.0092)},
    "cust_account_suspension_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<=", 0)},
    "cust_phone_number_change_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<=", 0)},
    "cust_mno_port_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<=", 0)},
    "cust_account_suspension_days_for_last_3_events": {"fraud": ("<=", 56), "normal": (">", 80.5)},
    "cust_account_suspension_interval_cv": {"fraud": (">=", 1.1198), "normal": ("<", 0.7245)},
    "cust_account_suspension_interval_stddev": {"fraud": ("<=", 35.95), "normal": (">", 76.9865)},
    "cust_account_suspension_k_event_velocity_3": {"fraud": ("<=", 0.0144), "normal": (">", 0.0484)},
    "cust_account_suspension_cumulative_event_slope": {"fraud": ("<=", 0.0078), "normal": (">", 0.0134)},
    "cust_account_suspension_interval_max": {"fraud": (">=", 179), "normal": ("<", 96)},
    "cust_account_suspension_interval_mean": {"fraud": ("<=", 23), "normal": (">", 52.3)},
    "cust_account_suspension_interval_median": {"fraud": ("<=", 15), "normal": (">", 33.5)},
    "cust_account_suspension_interval_min": {"fraud": ("<=", 9), "normal": (">", 30.5)},
    "cust_account_suspension_inverse_mean_interarrival_velocity": {"fraud": ("<=", 0.0084), "normal": (">", 0.01455)},
    "cust_account_suspension_monthly_cnt_std": {"fraud": ("<=", 0), "normal": (">", 0.1)},
    "cust_sim_change_days_for_last_5_events": {"fraud": ("<=", 63), "normal": (">", 127.5)},
    "cust_sim_change_k_event_velocity_5": {"fraud": ("<=", 0.0181), "normal": (">", 0.0253)},
    "cust_sim_change_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<", 0.8737)},
    "cust_account_suspension_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.2762)},
    "cust_phone_number_change_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.3447)},
    "cust_mno_port_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.5403)},
    "cust_account_suspension_cnt_30d_over_90d": {"fraud": (">=", 1), "normal": ("<", 0.4521)},
    "cust_device_change_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<", 0.7204)},
    "cust_account_suspension_active_months": {"fraud": (">=", 3), "normal": ("<", 2)},
    "cust_account_suspension_any_1d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_account_suspension_any_30d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_account_suspension_any_3d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_account_suspension_any_7d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_account_suspension_any_90d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_account_suspension_avg_monthly_cnt": {"fraud": (">=", 1.25), "normal": ("<", 1.08)},
    "cust_account_suspension_cnt_1d": {"fraud": (">=", 1), "normal": ("<", 0.018)},
    "cust_account_suspension_cnt_30d": {"fraud": (">=", 1), "normal": ("<", 0.4598)},
    "cust_account_suspension_cnt_3d": {"fraud": (">=", 1), "normal": ("<", 0.062)},
    "cust_account_suspension_cnt_7d": {"fraud": (">=", 1), "normal": ("<", 0.1338)},
    "cust_account_suspension_cnt_90d": {"fraud": (">=", 2), "normal": ("<", 1)},
    "cust_account_suspension_days_since_last": {"fraud": ("<=", 12.5), "normal": (">", 41)},
    "cust_account_suspension_event_day_of_week_entropy": {"fraud": (">=", 1.5), "normal": ("<", 0.7336)},
    "cust_account_suspension_event_hour_entropy": {"fraud": (">=", 1), "normal": ("<", 0.5833)},
    "cust_account_suspension_event_month_entropy": {"fraud": (">=", 1.5), "normal": ("<", 0.7726)},
    "cust_account_suspension_last_ts": {"fraud": ("<=", 29), "normal": (">", 61)},
    "cust_account_suspension_max_monthly_cnt": {"fraud": (">=", 2), "normal": ("<", 1.142)},
    "cust_account_suspension_monthly_peak_to_avg": {"fraud": (">=", 1.75), "normal": ("<", 1.0435)},
    "cust_account_suspension_recency_weighted_intensity": {"fraud": (">=", 0.176), "normal": ("<", 0.1312)},
    "cust_enstream_api_call_cnt_1d_over_3d": {"fraud": (">=", 1), "normal": ("<", 0.1667)},
    "cust_sim_change_k_event_velocity_3": {"fraud": ("<=", 0.015), "normal": (">", 0.0203)},
    "cust_sim_change_interval_cv": {"fraud": (">=", 1.4142), "normal": ("<", 1.2598)},
    "cust_sim_change_days_for_last_3_events": {"fraud": ("<=", 16), "normal": (">", 40)},
    "cust_sim_change_interval_stddev": {"fraud": ("<=", 4.2426), "normal": (">", 9.8995)},
    "cust_phone_number_change_cnt_30d_over_90d": {"fraud": (">=", 1), "normal": ("<", 0.5491)},
    "cust_sim_change_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.5)},
    "cust_mno_port_cnt_30d_over_90d": {"fraud": (">=", 1), "normal": ("<", 0.6822)},
    "cust_sim_change_monthly_cnt_std": {"fraud": (">=", 0.7071), "normal": ("<", 0.4082)},
    "cust_sim_change_cumulative_event_slope": {"fraud": ("<=", 0.0049), "normal": (">", 0.0069)},
    "cust_sim_change_inverse_mean_interarrival_velocity": {"fraud": ("<=", 0.0051), "normal": (">", 0.0071)},
    "cust_phone_number_change_any_1d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_phone_number_change_any_30d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_phone_number_change_any_3d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_phone_number_change_any_7d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_phone_number_change_any_90d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_phone_number_change_avg_monthly_cnt": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_phone_number_change_cnt_1d": {"fraud": (">=", 1), "normal": ("<", 0.0157)},
    "cust_phone_number_change_cnt_30d": {"fraud": (">=", 1), "normal": ("<", 0.2241)},
    "cust_phone_number_change_cnt_3d": {"fraud": (">=", 1), "normal": ("<", 0.0292)},
    "cust_phone_number_change_cnt_7d": {"fraud": (">=", 1), "normal": ("<", 0.0816)},
    "cust_phone_number_change_cnt_90d": {"fraud": (">=", 1), "normal": ("<", 0.3985)},
    "cust_phone_number_change_days_since_last": {"fraud": ("<=", 41), "normal": (">", 89)},
    "cust_phone_number_change_event_day_of_week_entropy": {"fraud": (">", 0), "normal": ("==", 0)},
    "cust_phone_number_change_event_hour_entropy": {"fraud": (">", 0), "normal": ("==", 0)},
    "cust_phone_number_change_last_ts": {"fraud": ("<=", 26), "normal": (">", 57)},
    "cust_phone_number_change_max_monthly_cnt": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_phone_number_change_recency_weighted_intensity": {"fraud": (">=", 0.1018), "normal": ("<", 0.0173)},
    "cust_sim_change_interval_max": {"fraud": ("<=", 7), "normal": (">", 10.5)},
    "cust_sim_change_interval_mean": {"fraud": ("<=", 7), "normal": (">", 10)},
    "cust_sim_change_interval_median": {"fraud": ("<=", 7), "normal": (">", 8)},
    "cust_sim_change_interval_min": {"fraud": ("<=", 2), "normal": (">", 4.5)},
    "cust_enstream_api_call_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.5)},
    "cust_sim_change_cnt_30d_over_90d": {"fraud": (">=", 1), "normal": ("<", 0.5)},
    "cust_device_change_k_event_velocity_5": {"fraud": (">=", 0.2336), "normal": ("<", 0.1163)},
    "cust_device_change_days_for_last_5_events": {"fraud": ("<=", 14), "normal": (">", 65)},
    "cust_mno_port_active_months": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_mno_port_any_1d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_mno_port_any_30d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_mno_port_any_3d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_mno_port_any_7d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_mno_port_any_90d": {"fraud": ("==", 1), "normal": ("==", 0)},
    "cust_mno_port_avg_monthly_cnt": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_mno_port_cnt_1d": {"fraud": (">=", 1), "normal": ("<", 0.0264)},
    "cust_mno_port_cnt_30d": {"fraud": (">=", 1), "normal": ("<", 0.2811)},
    "cust_mno_port_cnt_3d": {"fraud": (">=", 1), "normal": ("<", 0.0389)},
    "cust_mno_port_cnt_7d": {"fraud": (">=", 1), "normal": ("<", 0.1529)},
    "cust_mno_port_cnt_90d": {"fraud": (">=", 1), "normal": ("<", 0.4104)},
    "cust_mno_port_days_since_last": {"fraud": ("<=", 22), "normal": (">", 30)},
    "cust_mno_port_event_day_of_week_entropy": {"fraud": (">", 0), "normal": ("==", 0)},
    "cust_mno_port_event_hour_entropy": {"fraud": (">=", 1), "normal": ("<", 0.0114)},
    "cust_mno_port_event_month_entropy": {"fraud": (">", 0), "normal": ("==", 0)},
    "cust_mno_port_last_ts": {"fraud": ("<=", 38), "normal": (">", 53)},
    "cust_mno_port_max_monthly_cnt": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_mno_port_monthly_peak_to_avg": {"fraud": (">", 1), "normal": ("<=", 1)},
    "cust_mno_port_recency_weighted_intensity": {"fraud": (">=", 0.1747), "normal": ("<", 0.1132)},
    "cust_device_change_cnt_7d_over_30d": {"fraud": (">=", 1), "normal": ("<", 0.6165)},
}

OPS = {
    ">=": lambda c, v: c >= v,
    "<=": lambda c, v: c <= v,
    ">":  lambda c, v: c >  v,
    "<":  lambda c, v: c <  v,
    "==": lambda c, v: c == v,
}

def apply_rules(df, rules=RULES):
    """For each feature in rules, add <feature>_signal ∈ {fraud, normal, uncertain, NULL}."""
    for feat, rule in rules.items():
        if feat not in df.columns:
            continue
        col = F.col(feat)
        f_op, f_val = rule["fraud"]
        n_op, n_val = rule["normal"]
        fraud_cond  = OPS[f_op](col, f_val)
        normal_cond = OPS[n_op](col, n_val)
        df = df.withColumn(
            f"{feat}_signal",
            F.when(fraud_cond,  "fraud")
             .when(normal_cond, "normal")
             .when(col.isNotNull(), "uncertain")
             .otherwise(None)
        )
    return df

# Example: df_with_signals = apply_rules(df)


Connection: project.spark.compatibility | Run start time: 2026-06-08 16:28:25.790739 | Run duration : 0:00:00.267978s.


In [24]:
%%pyspark project.spark.compatibility
all_validation_signaled = apply_rules(all_validation)


Connection: project.spark.compatibility | Run start time: 2026-06-08 16:28:26.062304 | Run duration : 0:00:21.201103s.


In [25]:
%%pyspark project.spark.compatibility
row_count = all_validation_signaled.count()
col_count = len(all_validation_signaled.columns)
print(f"Rows: {row_count}")
print(f"Columns: {col_count}")


Rows: 1001588
Columns: 675

Connection: project.spark.compatibility | Run start time: 2026-06-08 16:28:47.267475 | Run duration : 0:00:05.332508s.


In [26]:
%%pyspark project.spark.compatibility
# ─── Feature groups by fraud type × confidence ─────────────────────
# Each list = features tagged H/M/L for that fraud type in the spreadsheet

FPF_H = [
    "cust_account_suspension_k_event_velocity_3",
    "cust_account_suspension_cumulative_event_slope",
    "cust_account_suspension_cnt_7d_over_30d",
    "cust_account_suspension_cnt_30d_over_90d",
    "cust_account_suspension_active_months",
    "cust_account_suspension_any_1d",
    "cust_account_suspension_any_30d",
    "cust_account_suspension_any_3d",
    "cust_account_suspension_any_7d",
    "cust_account_suspension_any_90d",
    "cust_account_suspension_cnt_30d",
    "cust_account_suspension_cnt_3d",
    "cust_account_suspension_cnt_7d",
    "cust_account_suspension_cnt_90d",
    "cust_account_suspension_days_since_last",
    "cust_account_suspension_last_ts",
    "cust_account_suspension_max_monthly_cnt",
    "cust_account_suspension_monthly_peak_to_avg",
    "cust_account_suspension_recency_weighted_intensity",
    "cust_enstream_api_call_cnt_7d_over_30d",
    "cust_sim_change_cnt_30d_over_90d",
]
FPF_M = [
    "cust_account_suspension_days_for_last_5_events",
    "cust_account_suspension_k_event_velocity_5",
    "cust_account_suspension_cnt_1d_over_3d",
    "cust_account_suspension_days_for_last_3_events",
    "cust_account_suspension_interval_mean",
    "cust_account_suspension_interval_median",
    "cust_account_suspension_interval_min",
    "cust_account_suspension_inverse_mean_interarrival_velocity",
    "cust_account_suspension_avg_monthly_cnt",
]
FPF_L = [
    "cust_enstream_api_call_cnt_1d_over_3d",
]

TrueName_H = [
    "cust_mno_port_cumulative_event_slope",
    "cust_device_change_cnt_1d_over_3d",
    "cust_enstream_api_call_cnt_1d_over_3d",
    "cust_enstream_api_call_cnt_7d_over_30d",
]
TrueName_M = [
    "cust_mno_port_interval_stddev",
    "cust_mno_port_cnt_30d_over_90d",
    "cust_mno_port_last_ts",
    "cust_mno_port_recency_weighted_intensity",
]
TrueName_L = [
    "cust_phone_number_change_interval_mean",
]

Synthetic_H = [
    "cust_sim_change_monthly_cnt_std",
    "cust_enstream_api_call_cnt_7d_over_30d",
]
Synthetic_M = [
    "cust_mno_port_interval_cv",
    "cust_mno_port_interval_mean",
    "cust_account_suspension_event_month_entropy",
    "cust_mno_port_cnt_30d_over_90d",
    "cust_mno_port_last_ts",
    "cust_mno_port_recency_weighted_intensity",
]
Synthetic_L = [
    "cust_phone_number_change_interval_mean",
]

Mule_H = [
    "cust_account_suspension_interval_cv",
    "cust_account_suspension_interval_stddev",
    "cust_sim_change_days_for_last_5_events",
    "cust_sim_change_k_event_velocity_5",
    "cust_sim_change_cnt_1d_over_3d",
    "cust_device_change_cnt_1d_over_3d",
    "cust_account_suspension_active_months",
    "cust_enstream_api_call_cnt_1d_over_3d",
    "cust_sim_change_interval_stddev",
    "cust_sim_change_monthly_cnt_std",
    "cust_sim_change_cumulative_event_slope",
    "cust_sim_change_inverse_mean_interarrival_velocity",
    "cust_phone_number_change_recency_weighted_intensity",
    "cust_sim_change_interval_mean",
    "cust_sim_change_interval_median",
    "cust_sim_change_interval_min",
    "cust_enstream_api_call_cnt_7d_over_30d",
    "cust_sim_change_cnt_30d_over_90d",
    "cust_device_change_k_event_velocity_5",
    "cust_device_change_days_for_last_5_events",
    "cust_device_change_cnt_7d_over_30d",
]
Mule_M = [
    "cust_mno_port_interval_stddev",
    "cust_phone_number_change_interval_stddev",
    "cust_account_suspension_days_since_last",
    "cust_sim_change_k_event_velocity_3",
    "cust_sim_change_interval_cv",
    "cust_sim_change_days_for_last_3_events",
    "cust_phone_number_change_cnt_30d_over_90d",
    "cust_mno_port_cnt_30d_over_90d",
    "cust_phone_number_change_active_months",
    "cust_phone_number_change_any_1d",
    "cust_phone_number_change_any_30d",
    "cust_phone_number_change_any_3d",
    "cust_phone_number_change_any_7d",
    "cust_phone_number_change_any_90d",
    "cust_phone_number_change_avg_monthly_cnt",
    "cust_phone_number_change_cnt_1d",
    "cust_phone_number_change_cnt_30d",
    "cust_phone_number_change_cnt_3d",
    "cust_phone_number_change_cnt_7d",
    "cust_phone_number_change_cnt_90d",
    "cust_phone_number_change_days_since_last",
    "cust_phone_number_change_event_month_entropy",
    "cust_phone_number_change_last_ts",
    "cust_mno_port_active_months",
    "cust_mno_port_any_1d",
    "cust_mno_port_any_30d",
    "cust_mno_port_any_3d",
    "cust_mno_port_any_7d",
    "cust_mno_port_any_90d",
    "cust_mno_port_cnt_1d",
    "cust_mno_port_cnt_30d",
    "cust_mno_port_cnt_3d",
    "cust_mno_port_cnt_7d",
    "cust_mno_port_cnt_90d",
    "cust_mno_port_days_since_last",
    "cust_mno_port_event_month_entropy",
    "cust_mno_port_last_ts",
    "cust_mno_port_recency_weighted_intensity",
]
Mule_L = [
    "cust_phone_number_change_interval_mean",
]

ATO_H = [
    "cust_mno_port_cumulative_event_slope",
    "cust_mno_port_inverse_mean_interarrival_velocity",
    "cust_mno_port_k_event_velocity_3",
    "cust_mno_port_k_event_velocity_5",
    "cust_phone_number_change_cumulative_event_slope",
    "cust_phone_number_change_inverse_mean_interarrival_velocity",
    "cust_phone_number_change_k_event_velocity_3",
    "cust_phone_number_change_k_event_velocity_5",
    "cust_phone_number_change_cnt_1d_over_3d",
    "cust_mno_port_cnt_1d_over_3d",
    "cust_sim_change_days_for_last_5_events",
    "cust_sim_change_k_event_velocity_5",
    "cust_sim_change_cnt_1d_over_3d",
    "cust_device_change_cnt_1d_over_3d",
    "cust_enstream_api_call_cnt_1d_over_3d",
    "cust_sim_change_k_event_velocity_3",
    "cust_sim_change_interval_cv",
    "cust_sim_change_days_for_last_3_events",
    "cust_sim_change_interval_stddev",
    "cust_sim_change_cnt_7d_over_30d",
    "cust_sim_change_monthly_cnt_std",
    "cust_sim_change_cumulative_event_slope",
    "cust_sim_change_inverse_mean_interarrival_velocity",
    "cust_phone_number_change_any_1d",
    "cust_phone_number_change_any_30d",
    "cust_phone_number_change_any_3d",
    "cust_phone_number_change_any_7d",
    "cust_phone_number_change_any_90d",
    "cust_phone_number_change_cnt_1d",
    "cust_phone_number_change_cnt_30d",
    "cust_phone_number_change_cnt_3d",
    "cust_phone_number_change_cnt_7d",
    "cust_phone_number_change_cnt_90d",
    "cust_phone_number_change_days_since_last",
    "cust_phone_number_change_recency_weighted_intensity",
    "cust_sim_change_interval_mean",
    "cust_sim_change_interval_median",
    "cust_sim_change_interval_min",
    "cust_enstream_api_call_cnt_7d_over_30d",
    "cust_sim_change_cnt_30d_over_90d",
    "cust_mno_port_any_1d",
    "cust_mno_port_any_30d",
    "cust_mno_port_any_3d",
    "cust_mno_port_any_7d",
    "cust_mno_port_any_90d",
    "cust_mno_port_cnt_1d",
    "cust_mno_port_cnt_30d",
    "cust_mno_port_cnt_3d",
    "cust_mno_port_cnt_7d",
    "cust_mno_port_cnt_90d",
    "cust_mno_port_days_since_last",
    "cust_mno_port_recency_weighted_intensity",
]
ATO_M = [
    "cust_mno_port_interval_cv",
    "cust_mno_port_interval_mean",
    "cust_phone_number_change_interval_cv",
    "cust_phone_number_change_cnt_30d_over_90d",
    "cust_mno_port_cnt_30d_over_90d",
    "cust_device_change_cnt_7d_over_30d",
]
ATO_L = []  # (empty)



Connection: project.spark.compatibility | Run start time: 2026-06-08 16:28:52.605546 | Run duration : 0:00:01.317398s.


In [27]:
%%pyspark project.spark.compatibility
# Combined H/M/L: dedup + highest tier wins
all_H = set(FPF_H + TrueName_H + Synthetic_H + Mule_H + ATO_H)
all_M = set(FPF_M + TrueName_M + Synthetic_M + Mule_M + ATO_M)
all_L = set(FPF_L + TrueName_L + Synthetic_L + Mule_L + ATO_L)
Combined_H = sorted(all_H)
Combined_M = sorted(all_M - all_H)
Combined_L = sorted(all_L - all_H - all_M)
print(f"Combined: H={len(Combined_H)} M={len(Combined_M)} L={len(Combined_L)}")

GROUPS = {
    "FPF":       {"H": FPF_H,       "M": FPF_M,       "L": FPF_L},
    "TrueName":  {"H": TrueName_H,  "M": TrueName_M,  "L": TrueName_L},
    "Synthetic": {"H": Synthetic_H, "M": Synthetic_M, "L": Synthetic_L},
    "Mule":      {"H": Mule_H,      "M": Mule_M,      "L": Mule_L},
    "ATO":       {"H": ATO_H,       "M": ATO_M,       "L": ATO_L},
    "Combined":  {"H": Combined_H,  "M": Combined_M,  "L": Combined_L},
}

WEIGHTS_FRAUD = {"H": 3, "M": 2, "L": 1}

def add_type_scores_both(df, type_name, levels_dict):
    hits_fo, score_fo = F.lit(0), F.lit(0.0)
    hits_wu, score_wu = F.lit(0), F.lit(0.0)
    for level, feats in levels_dict.items():
        w_f = WEIGHTS_FRAUD[level]
        w_u = w_f * 0.5
        for feat in feats:
            sig = f"{feat}_signal"
            if sig not in df.columns:
                continue
            is_f = F.when(F.col(sig) == "fraud",     1).otherwise(0)
            is_u = F.when(F.col(sig) == "uncertain", 1).otherwise(0)
            hits_fo  = hits_fo  + is_f
            score_fo = score_fo + is_f * w_f
            hits_wu  = hits_wu  + is_f + is_u
            score_wu = score_wu + is_f * w_f + is_u * w_u
    return (df
        .withColumn(f"{type_name}_hits",                 hits_fo)
        .withColumn(f"{type_name}_score",                score_fo)
        .withColumn(f"{type_name}_hits_with_uncertain",  hits_wu)
        .withColumn(f"{type_name}_score_with_uncertain", score_wu))

df_scored = all_validation_signaled
for type_name, levels in GROUPS.items():
    df_scored = add_type_scores_both(df_scored, type_name, levels)

score_cols = (
    [f"{t}_hits"                 for t in GROUPS.keys()] +
    [f"{t}_score"                for t in GROUPS.keys()] +
    [f"{t}_hits_with_uncertain"  for t in GROUPS.keys()] +
    [f"{t}_score_with_uncertain" for t in GROUPS.keys()]
)
df_scored.select(*score_cols).describe().show()


Combined: H=76 M=24 L=1
+-------+-------------------+--------------------+--------------------+-------------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+-----------------+-----------------------+----------------------------+-----------------------------+------------------------+-----------------------+----------------------------+------------------------+-----------------------------+------------------------------+-------------------------+------------------------+-----------------------------+
|summary|           FPF_hits|       TrueName_hits|      Synthetic_hits|          Mule_hits|           ATO_hits|     Combined_hits|          FPF_score|    TrueName_score|    Synthetic_score|        Mule_score|         ATO_score|   Combined_score|FPF_hits_with_uncertain|TrueName_hits_with_uncertain|Synthetic_hits_with_uncertain|Mule_hits_with_uncertain|ATO_hits_with_uncertain|Combined_hits_with_uncert

In [28]:
%%pyspark project.spark.compatibility
OUT_PATH = "s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2/validation_final"
df_scored.write.mode("overwrite").parquet(OUT_PATH)
print(f"saved scored → {OUT_PATH}")
print(f"rows: {df_scored.count():,}, cols: {len(df_scored.columns)}")

saved scored → s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2/validation_final
rows: 1,001,588, cols: 699

Connection: project.spark.compatibility | Run start time: 2026-06-08 16:29:26.045911 | Run duration : 0:00:42.750722s.


In [29]:
%%pyspark project.spark.compatibility
df_scored = spark.read.parquet('s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2/validation_final/')


Connection: project.spark.compatibility | Run start time: 2026-06-08 16:30:08.800613 | Run duration : 0:00:01.828698s.


In [30]:
%%pyspark project.spark.compatibility
df_scored.printSchema()

root
 |-- cust_phone_number_change_to_status_change_s_avg_transition_days: string (nullable = true)
 |-- cust_status_change_a_to_phone_number_change_avg_transition_days: string (nullable = true)
 |-- cust_sim_change_to_device_change_transition_velocity: string (nullable = true)
 |-- cust_device_change_to_enstream_api_call_transition_velocity: string (nullable = true)
 |-- cust_status_change_c_to_status_change_a_transition_velocity: string (nullable = true)
 |-- cust_status_change_s_to_status_change_c_transition_velocity: string (nullable = true)
 |-- cust_status_change_s_to_enstream_api_call_min_transition_days: string (nullable = true)
 |-- cust_phone_number_change_to_device_change_transition_velocity: string (nullable = true)
 |-- cust_enstream_api_call_to_device_change_avg_transition_days: string (nullable = true)
 |-- cust_status_change_s_to_phone_number_change_min_transition_days: string (nullable = true)
 |-- cust_device_change_to_sim_change_min_transition_days: string (nullable 

In [31]:
%%pyspark project.spark.compatibility
df_scored.count()



1001588
Connection: project.spark.compatibility | Run start time: 2026-06-08 16:30:13.727185 | Run duration : 0:00:03.270456s.


In [32]:
%%pyspark project.spark.compatibility



Connection: project.spark.compatibility | Run start time: 2026-06-08 16:30:17.001540 | Run duration : 0:00:00.524407s.


In [33]:
%%pyspark project.spark.compatibility



Connection: project.spark.compatibility | Run start time: 2026-06-08 16:30:17.529541 | Run duration : 0:00:00.536883s.


In [37]:
%%pyspark project.spark.compatibility
from pyspark.sql import functions as F
RAW  = "option5_reservoir_sampled_imf_raw_score"
ANOM = "option5_reservoir_sampled_imf_anomaly_score"

for col in [RAW, ANOM]:
    for cls, cond in [("NORMAL", F.col("fraud_type") == "non_fraud"),
                      ("FRAUD",  F.col("fraud_type") != "non_fraud")]:
        sub = df_scored.filter(cond)
        nd  = sub.select(col).distinct().count()
        print(f"\n===== {col} | {cls} | number of distinct values = {nd} =====")
        (sub.groupBy(F.round(F.col(col), 4).alias("value"))
            .count()
            .orderBy(F.desc("count"))
            .show(15, truncate=False))



===== option5_reservoir_sampled_imf_raw_score | NORMAL | 不同值个数 distinct = 1 =====
+------+-------+
|value |count  |
+------+-------+
|0.1642|1000000|
+------+-------+


===== option5_reservoir_sampled_imf_raw_score | FRAUD | 不同值个数 distinct = 1035 =====
+------+-----+
|value |count|
+------+-----+
|0.1361|162  |
|0.6144|58   |
|0.3693|50   |
|0.2646|15   |
|0.2372|14   |
|0.3608|11   |
|0.3208|10   |
|0.3205|9    |
|0.3429|8    |
|0.3416|8    |
|0.3652|8    |
|0.2377|6    |
|0.2278|6    |
|0.3342|6    |
|0.541 |6    |
+------+-----+
only showing top 15 rows


===== option5_reservoir_sampled_imf_anomaly_score | NORMAL | 不同值个数 distinct = 1 =====
+-----+-------+
|value|count  |
+-----+-------+
|0.5  |1000000|
+-----+-------+


===== option5_reservoir_sampled_imf_anomaly_score | FRAUD | 不同值个数 distinct = 669 =====
+------+-----+
|value |count|
+------+-----+
|1.0   |499  |
|0.0   |162  |
|0.5231|50   |
|0.2882|15   |
|0.2268|14   |
|0.504 |11   |
|0.4145|10   |
|0.4137|9    |
|0.464 |8    |

In [35]:
%%pyspark project.spark.compatibility
for col in ["Combined_score", "Combined_score_with_uncertain"]:
    print("===", col, "===")
    (df_scored
     .withColumn("y", F.when(F.col("fraud_type") == "non_fraud", "normal").otherwise("fraud"))
     .groupBy("y")
     .agg(F.round(F.avg(col), 3).alias("avg"),
          F.round(F.stddev(col), 3).alias("std"),
          F.round(F.max(col), 1).alias("max"),
          F.round(F.avg((F.col(col) > 0).cast("int")), 3).alias("frac_gt0"))  # fraction with signal
     .show())


=== Combined_score ===
+------+-----+-----+----+--------+
|     y|  avg|  std| max|frac_gt0|
+------+-----+-----+----+--------+
|normal|0.981|3.713|72.0|    0.12|
| fraud|0.574|1.981|15.0|    0.12|
+------+-----+-----+----+--------+

=== Combined_score_with_uncertain ===
+------+-----+-----+----+--------+
|     y|  avg|  std| max|frac_gt0|
+------+-----+-----+----+--------+
|normal|1.293|4.154|83.5|   0.248|
| fraud|0.588|2.002|15.0|   0.125|
+------+-----+-----+----+--------+
Connection: project.spark.compatibility | Run start time: 2026-06-08 16:30:18.604951 | Run duration : 0:00:04.411478s.


In [36]:
%%pyspark project.spark.compatibility
import boto3
s3 = boto3.client("s3"); bucket = "enstream-lake-gold-dev"
RUN = ("poc/analysis/imf_real_data_validation/"
       "run_id=imf_real_data_validation_bugfixed_default_available_feature_requests_20260603_172459")
def ls(p):
    p = p.rstrip("/") + "/"
    r = s3.list_objects_v2(Bucket=bucket, Prefix=p, Delimiter="/")
    print(f"--- {p}")
    for x in r.get("CommonPrefixes", []): print("  [dir] ", x["Prefix"].split("/")[-2])
    for x in r.get("Contents", [])[:8]:   print("  [file]", x["Key"].split("/")[-1])
ls(f"{RUN}/scored")
ls(f"{RUN}/prepared")


--- poc/analysis/imf_real_data_validation/run_id=imf_real_data_validation_bugfixed_default_available_feature_requests_20260603_172459/scored/
  [file] nbc_scored.csv
  [file] nov_background_scored_sample_10k.csv
  [file] phase1_scored.csv
--- poc/analysis/imf_real_data_validation/run_id=imf_real_data_validation_bugfixed_default_available_feature_requests_20260603_172459/prepared/
  [dir]  fraud_nbc_matrix.parquet
  [dir]  fraud_phase1_matrix.parquet
  [dir]  nonfraud_test_dt=2024-10-01_00-00-00_n=1000000.parquet
  [dir]  nonfraud_train_dt=2024-03-01_00-00-00.parquet
  [dir]  nonfraud_train_dt=2024-04-01_00-00-00.parquet
  [dir]  nonfraud_train_dt=2024-05-01_00-00-00.parquet
  [dir]  nonfraud_train_dt=2024-06-01_00-00-00.parquet
  [dir]  nonfraud_train_dt=2024-07-01_00-00-00.parquet
  [dir]  nonfraud_train_dt=2024-08-01_00-00-00.parquet
  [dir]  nonfraud_train_dt=2024-09-01_00-00-00.parquet
  [dir]  nonfraud_train_dt=2024-10-01_00-00-00.parquet

Connection: project.spark.compatibility |